<a href="https://colab.research.google.com/github/Kartik-bsc/data_science_assignment/blob/main/international_t20_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('International_T20_Data.csv')

In [ ]:
df.head()

,innings,meta.data_version,meta.created,meta.revision,info.dates,info.gender,info.match_type,info.outcome.by.wickets,info.outcome.winner,info.overs,...,info.outcome.by.runs,info.match_type_number,info.neutral_venue,info.outcome.method,info.outcome.result,info.outcome.eliminator,info.supersubs.New Zealand,info.supersubs.South Africa,info.bowl_out,info.outcome.bowl_out
0,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-18,2,"[datetime.date(2017, 2, 17)]",male,T20,5.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-19,2,"[datetime.date(2017, 2, 19)]",male,T20,2.0,Sri Lanka,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'1st innings': {'team': 'Australia', 'delive...",0.9,2017-02-23,1,"[datetime.date(2017, 2, 22)]",male,T20,NaN,Australia,20,...,41.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'1st innings': {'team': 'Hong Kong', 'delive...",0.9,2016-09-12,1,"[datetime.date(2016, 9, 5)]",male,T20,NaN,Hong Kong,20,...,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'1st innings': {'team': 'Zimbabwe', 'deliver...",0.9,2016-06-19,1,"[datetime.date(2016, 6, 18)]",male,T20,NaN,Zimbabwe,20,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Rename all the column names to their appropriate names, for example meta.created should be renamed as created_date

In [ ]:
df.rename(columns={
    'created': 'created_date',
    'outcome.winner': 'winner',
    'outcome.result': 'result',
    'toss.decision': 'toss_decision',
    'toss.winner': 'toss_winner',
    'player_of_match': 'player_of_match'
}, inplace=True)

# Find out the top three venues which hosted the greatest number of matches.

In [8]:
top_3_venues = df['venue'].value_counts().head(3)

print("Top 3 Venues by Matches Hosted:")
print(top_3_venues)

# Find out the pair of cricket teams who played the most number of T20 matches against each other.

In [8]:
def parse_and_sort_teams(team_string):
    # Convert string like "['Team A', 'Team B']" into a real Python list
    team_list = ast.literal_eval(team_string)
    # Sort alphabetically and convert to a tuple so it can be counted
    return tuple(sorted(team_list))

# Apply the function to create a matchup column
df['matchup'] = df['teams'].apply(parse_and_sort_teams)

# Find the most frequent pair
top_matchup = df['matchup'].value_counts().head(1)

print("The pair of teams that played the most against each other:")
print(top_matchup)

# Print the top five teams by their win percentages. Win percentage is defined as the number of matches won divided by the number of matches played and then multiplied by 100.

In [ ]:
# 1. Convert the 'teams' string into a real list for the whole column
df['team_list'] = df['teams'].apply(ast.literal_eval)

# 2. Explode the list so each team gets its own row, then count matches played
matches_played = df.explode('team_list')['team_list'].value_counts()

# 3. Count matches won by each team
matches_won = df['winner'].value_counts()

# 4. Calculate Win Percentage (Wins / Played * 100)
win_percentage = (matches_won / matches_played) * 100

# 5. Drop NaN values (teams with no wins or no result matches) and sort
top_5_teams = win_percentage.dropna().sort_values(ascending=False).head(5)

print("Top 5 Teams by Win Percentage:")
print(top_5_teams)

Write a function to get the scorecard of each match. This function would take the innings value as argument and return two scorecard dataframes each for one team as shown below. So the first dataframe would contain the top 4 scorers of the team who batted first and the top 4 bowlers of the opponent team. And the second dataframe would contain the top 4 scorers of the team who batted second and the top 4 bowlers of the opponent team.
## New section

In [ ]:


import pandas as pd
import ast
from IPython.display import display # This makes dataframes look beautiful in Colab!

def get_scorecard(innings_value):
    """
    Takes the nested innings string from a single row and returns
    two dataframes formatted like the TV broadcast scorecard.
    """
    # 1. Parse the string into a Python list/dict safely
    if isinstance(innings_value, str):
        try:
            innings_data = ast.literal_eval(innings_value)
        except (ValueError, SyntaxError):
            return None, None
    else:
        innings_data = innings_value

    scorecards = []

    # 2. Iterate over 1st and 2nd innings
    for inn in innings_data:
        inn_key = list(inn.keys())[0] # Usually '1st innings' or '2nd innings'
        deliveries = inn[inn_key]['deliveries']

        batting_stats = {}
        bowling_stats = {}

        # 3. Loop through every single ball
        for delivery in deliveries:
            ball_key = list(delivery.keys())[0]
            ball_data = delivery[ball_key]

            batsman = ball_data.get('batsman')
            bowler = ball_data.get('bowler')

            # Extract runs
            runs_bat = ball_data.get('runs', {}).get('batsman', 0)
            runs_total = ball_data.get('runs', {}).get('total', 0)
            extras = ball_data.get('extras', {})

            # --- BATTING AGGREGATIONS ---
            if batsman not in batting_stats:
                batting_stats[batsman] = {'Runs': 0, 'Balls': 0}

            batting_stats[batsman]['Runs'] += runs_bat

            # Wides don't count as a ball faced for the batsman
            if 'wides' not in extras:
                batting_stats[batsman]['Balls'] += 1

            # --- BOWLING AGGREGATIONS ---
            if bowler not in bowling_stats:
                bowling_stats[bowler] = {'Wickets': 0, 'Runs_Conceded': 0, 'Legal_Balls': 0}

            # Bowler doesn't concede byes or legbyes
            bowler_runs = runs_total
            if 'byes' in extras or 'legbyes' in extras:
                bowler_runs -= (extras.get('byes', 0) + extras.get('legbyes', 0))
            bowling_stats[bowler]['Runs_Conceded'] += bowler_runs

            # Calculate legal deliveries for overs (no wides, no no-balls)
            if 'wides' not in extras and 'noballs' not in extras:
                bowling_stats[bowler]['Legal_Balls'] += 1

            # Calculate wickets (excluding run outs)
            if 'wicket' in ball_data:
                kind = ball_data['wicket'].get('kind', '')
                if kind not in ['run out', 'retired hurt', 'obstructing the field']:
                    bowling_stats[bowler]['Wickets'] += 1

        # 4. Format the Batting DataFrame
        bat_df = pd.DataFrame.from_dict(batting_stats, orient='index').reset_index()
        bat_df.rename(columns={'index': 'Batsman'}, inplace=True)
        bat_df = bat_df.sort_values(by=['Runs', 'Balls'], ascending=[False, True]).head(4).reset_index(drop=True)

        # 5. Format the Bowling DataFrame
        bowl_df = pd.DataFrame.from_dict(bowling_stats, orient='index').reset_index()
        if not bowl_df.empty:
            bowl_df.rename(columns={'index': 'Bowler'}, inplace=True)

            # Create standard TV format: 'Wickets-Runs' and 'Overs'
            bowl_df['W-R'] = bowl_df['Wickets'].astype(str) + '-' + bowl_df['Runs_Conceded'].astype(str)
            bowl_df['Overs'] = (bowl_df['Legal_Balls'] // 6).astype(str) + '.' + (bowl_df['Legal_Balls'] % 6).astype(str)
            bowl_df['Overs'] = bowl_df['Overs'].str.replace('.0', '', regex=False) # Clean up "4.0" to just "4"

            # Sort by Wickets, then least runs conceded
            bowl_df = bowl_df.sort_values(by=['Wickets', 'Runs_Conceded'], ascending=[False, True]).head(4).reset_index(drop=True)
            bowl_df = bowl_df[['Bowler', 'W-R', 'Overs']] # Keep only the TV columns
        else:
            bowl_df = pd.DataFrame(columns=['Bowler', 'W-R', 'Overs'])

        # 6. Combine Side-by-Side
        scorecard = pd.concat([bat_df, bowl_df], axis=1)
        scorecard = scorecard.fillna('') # Clean up any NaNs if there are fewer than 4 players
        scorecards.append(scorecard)

    # Catch edge cases (e.g., match washed out before 2nd innings)
    while len(scorecards) < 2:
        scorecards.append(pd.DataFrame(columns=['Batsman', 'Runs', 'Balls', 'Bowler', 'W-R', 'Overs']))

    return scorecards[0], scorecards[1]


# =====================================================================
# EXECUTION BLOCK - This is the part that actually runs and prints it!
# =====================================================================

# Grab the innings data from the very first row (index 0) of your dataframe
sample_match_innings = df.loc[0, 'innings']

# Run the function on that data
innings_1_df, innings_2_df = get_scorecard(sample_match_innings)

# Display the output using Colab's rich display format
print("=================== 1ST INNINGS ===================")
display(innings_1_df)

print("\n=================== 2ND INNINGS ===================")
display(innings_2_df)